In [15]:
import requests
import pandas as pd
import re
import argparse
from typing import Any
import time

In [17]:
EARCH_URL = "https://publicity.businessportal.gr/api/search"
DETAILS_URL = "https://publicity.businessportal.gr/api/company/details"

DEFAULT_HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9,el;q=0.8",
    "Content-Type": "application/json",
    "Origin": "https://publicity.businessportal.gr",
    "Referer": "https://publicity.businessportal.gr/",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
}


def build_search_payload(company_name: str, language: str = "el") -> dict[str, Any]:
    return {
        "dataToBeSent": {
            "inputField": company_name,
            "city": None,
            "postcode": None,
            "legalType": [],
            "status": [],
            "suspension": [],
            "category": [],
            "specialCharacteristics": [],
            "employeeNumber": [],
            "armodiaGEMI": [],
            "kad": [],
            "recommendationDateFrom": None,
            "recommendationDateTo": None,
            "closingDateFrom": None,
            "closingDateTo": None,
            "alterationDateFrom": None,
            "alterationDateTo": None,
            "person": [],
            "personrecommendationDateFrom": None,
            "personrecommendationDateTo": None,
            "radioValue": "all",
            "places": [],
        },
        "token": None,
        "language": language,
    }


def search_gemi(company_name: str, language: str = "el") -> dict[str, Any]:
    response = requests.post(
        SEARCH_URL,
        headers=DEFAULT_HEADERS,
        json=build_search_payload(company_name, language=language),
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def get_company_hits(company_name: str, language: str = "el") -> list[dict[str, Any]]:
    data = search_gemi(company_name, language=language)
    return data.get("company", {}).get("hits", [])


def get_company_details_by_gemi(
    gemi_number: str, language: str = "el"
) -> dict[str, Any]:
    headers = {
        **DEFAULT_HEADERS,
        "Referer": f"https://publicity.businessportal.gr/company/{gemi_number}",
    }
    payload = {"query": {"arGEMI": gemi_number}, "token": None, "language": language}
    response = requests.post(
        DETAILS_URL,
        headers=headers,
        json=payload,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def get_company_details_by_afm(
    afm: str, language: str = "el", delay_seconds: float = 2
) -> dict[str, Any]:
    hits = get_company_hits(afm, language=language)
    if not hits:
        raise LookupError(f"No company found for AFM {afm}")

    exact_hit = next((hit for hit in hits if hit.get("afm") == afm), None)
    if exact_hit is None:
        raise LookupError(f"No exact AFM match found for {afm}")

    gemi_number = exact_hit.get("gemiNumber")
    if not gemi_number:
        raise LookupError(f"Search hit for AFM {afm} does not include gemiNumber")

    time.sleep(delay_seconds)

    return {
        "search_hit": exact_hit,
        "details": get_company_details_by_gemi(gemi_number, language=language),
    }


def extract_company_profile(result: dict[str, Any]) -> dict[str, Any]:
    search_hit = result.get("search_hit", {})
    payload = result["details"]["companyInfo"]["payload"]
    company = payload["company"]
    titles = payload.get("titles", [])
    management_persons = payload.get("managementPersons", [])
    representation = payload.get("representation", [])
    kad_data = payload.get("kadData", [])

    current_management = [
        {
            "first_name": person.get("firstName"),
            "last_name": person.get("lastName"),
            "afm": person.get("afm"),
            "capacity": person.get("capacityDescr"),
            "date_from": person.get("dateFrom"),
            "date_to": person.get("dateTo"),
            "active": person.get("active"),
            "shared": person.get("shared"),
            "non_shared": person.get("nonShared"),
        }
        for person in management_persons
        if person.get("active") == 1 and person.get("tableName") != "member"
    ]

    current_ownership = [
        {
            "first_name": person.get("firstName"),
            "last_name": person.get("lastName"),
            "afm": person.get("afm"),
            "capacity": person.get("capacityDescr"),
            "percentage": person.get("percentage"),
            "date_from": person.get("dateFrom"),
            "date_to": person.get("dateTo"),
            "active": person.get("active"),
        }
        for person in management_persons
        if person.get("active") == 1 and person.get("tableName") == "member"
    ]

    current_representation = [
        {
            "name": person.get("name"),
            "afm": person.get("afm"),
            "shared": person.get("shared"),
            "non_shared": person.get("nonShared"),
            "active": person.get("active"),
            "active_from": person.get("activeFrom"),
            "active_to": person.get("activeTo"),
        }
        for person in representation
        if person.get("active") == 1
    ]

    current_titles = [
        {
            "title": item.get("title"),
            "title_i18n": [x.get("title") for x in item.get("titleI18n", [])],
            "is_enable": item.get("isEnable"),
        }
        for item in titles
        if item.get("isEnable") == 1
    ]

    inactive_titles = [
        {
            "title": item.get("title"),
            "title_i18n": [x.get("title") for x in item.get("titleI18n", [])],
            "is_enable": item.get("isEnable"),
        }
        for item in titles
        if item.get("isEnable") != 1
    ]

    main_kads = [
        {"kad": item.get("kad"), "description": item.get("descr")}
        for item in kad_data
        if item.get("activities") == "Κύρια"
    ]

    secondary_kads = [
        {"kad": item.get("kad"), "description": item.get("descr")}
        for item in kad_data
        if item.get("activities") != "Κύρια"
    ]

    return {
        "core_company_identity": {
            "name": company.get("name"),
            "name_i18n": company.get("namei18n"),
            "afm": company.get("afm"),
            "gemi_number": company.get("id", "").lstrip("0") or company.get("id"),
            "company_id": company.get("id"),
            "legal_type": company.get("legalType", {}).get("desc"),
            "branch_type": company.get("branchType"),
            "is_branch": company.get("isABranch"),
            "is_foreign_branch": company.get("isForeignBranch"),
        },
        "name_variations": {
            "current_titles": current_titles,
            "historical_or_disabled_titles": inactive_titles,
            "search_titles": search_hit.get("title", []),
        },
        "address": {
            "street": company.get("company_street"),
            "street_number": company.get("company_street_number"),
            "city": company.get("company_city"),
            "municipality": company.get("company_municipality"),
            "region": company.get("company_region"),
            "zip_code": company.get("company_zip_code"),
            "full_search_address": search_hit.get("addressCity"),
        },
        "dates": {
            "date_start": company.get("dateStart"),
            "date_gemi_registered": company.get("dateGemiRegistered"),
        },
        "management": {"current": current_management},
        "ownership": {"current_members": current_ownership},
        "representation": {"current": current_representation},
        "contact": {
            "telephone": payload.get("moreInfo", {}).get("telephone"),
            "email": payload.get("moreInfo", {}).get("email"),
            "website": company.get("companyWebsite"),
            "eshop": company.get("companyEshop"),
        },
        "activities": {
            "main_kad": main_kads,
            "secondary_kads": secondary_kads,
        },
        "status": {
            "current_status": company.get("companyStatus", {}).get("status"),
            "is_suspended": company.get("issuspended"),
            "company_status_history": [
                {"date": item.get("dt"), "status": item.get("descr")}
                for item in payload.get("companyStatusHistoryInfo", [])
            ],
            "corporate_status_history": [
                {"date": item.get("dt"), "legal_form": item.get("descr")}
                for item in payload.get("corporateStatusHistoryInfo", [])
            ],
        },
    }


def get_company_profile_by_afm(
    afm: str, language: str = "el", delay_seconds: float = 2
) -> dict[str, Any]:
    return extract_company_profile(
        get_company_details_by_afm(
            afm,
            language=language,
            delay_seconds=delay_seconds,
        )
    )


def get_company_profile_by_gemi(gemi_number: str, language: str = "el") -> dict[str, Any]:
    details = get_company_details_by_gemi(gemi_number, language=language)
    result = {"search_hit": {"gemiNumber": gemi_number, "title": []}, "details": details}
    return extract_company_profile(result)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Query the GEMI search endpoint extracted from the workshop notebook."
    )
    parser.add_argument(
        "query",
        nargs="?",
        help="Company name or AFM to search for",
    )
    parser.add_argument(
        "--language",
        default="el",
        choices=["el", "en"],
        help="Response language sent in the request payload",
    )
    parser.add_argument(
        "--details-by-gemi",
        metavar="GEMI_NUMBER",
        help="Fetch full company details using a GEMI number",
    )
    parser.add_argument(
        "--details-by-afm",
        metavar="AFM",
        help="Search by AFM and then fetch the full company details",
    )
    args = parser.parse_args()

    if args.details_by_gemi:
        print(get_company_details_by_gemi(args.details_by_gemi, language=args.language))
        return

    if args.details_by_afm:
        print(get_company_details_by_afm(args.details_by_afm, language=args.language))
        return

    if not args.query:
        parser.error("Provide a query, --details-by-gemi, or --details-by-afm.")

    hits = get_company_hits(args.query, language=args.language)
    for hit in hits[:10]:
        print(
            {
                "name": hit.get("name"),
                "gemiNumber": hit.get("gemiNumber"),
                "afm": hit.get("afm"),
                "status": hit.get("status"),
                "legalType": hit.get("legalType"),
            }
        )

In [19]:
get_company_profile_by_afm("043170596")

{'core_company_identity': {'name': 'ΤΙΓΚΑΣ ΚΩΝΣΤΑΝΤΙΝΟΣ ΤΟΥ ΑΝΤΩΝΙΟΥ',
  'name_i18n': 'TIGKAS KONSTANTINOS TOU ANTONIOU',
  'afm': '043170596',
  'gemi_number': '53257348000',
  'company_id': '053257348000',
  'legal_type': 'ΑΤΟΜΙΚΗ',
  'branch_type': 'Ελληνικής Επιχείρησης',
  'is_branch': False,
  'is_foreign_branch': False},
 'name_variations': {'current_titles': [],
  'historical_or_disabled_titles': [],
  'search_titles': []},
 'address': {'street': 'ΑΓΙΑΣ ΤΡΙΑΔΟΣ & ΑΝΔΡΟΥΤΣΟΥ 12',
  'street_number': '',
  'city': 'ΚΑΤΕΡΙΝΗ',
  'municipality': 'ΚΑΤΕΡΙΝΗΣ / ΠΙΕΡΙΑΣ',
  'region': '',
  'zip_code': '60100',
  'full_search_address': 'ΑΓΙΑΣ ΤΡΙΑΔΟΣ & ΑΝΔΡΟΥΤΣΟΥ 12 , ΚΑΤΕΡΙΝΗ, 60100'},
 'dates': {'date_start': '04/07/1988', 'date_gemi_registered': '10/07/2023'},
 'management': {'current': []},
 'ownership': {'current_members': []},
 'representation': {'current': []},
 'contact': {'telephone': '2351037866',
  'email': 'ktigkas@gmail.com',
  'website': None,
  'eshop': None},
 'activities

In [12]:
get_company_details_by_gemi("20805730000")

{'companyInfo': {'errorCode': None,
  'errorMessage': None,
  'payload': {'company': {'id': '020805730000',
    'name': 'ΒΙΟΣ ΑΝΩΝΥΜΗ ΕΤΑΙΡΕΙΑ',
    'branchParentOrigin': {'branchType': None,
     'parent_company_id': None,
     'parent_co_name': None},
    'issuspended': None,
    'namei18n': 'VIOS ANONYMI ETAIREIA',
    'afm': '998342580',
    'companyStatus': {'id': 1, 'status': 'Ενεργή'},
    'dateStart': '23/06/2009',
    'legalType': {'id': '1', 'desc': 'ΑΕ'},
    'branchType': 'Ελληνικής Επιχείρησης',
    'dateGemiRegistered': '23/06/2009',
    'company_city': 'ΚΑΒΑΛΑ',
    'company_street': 'Μ. ΑΛΕΞΑΝΔΡΟΥ',
    'company_street_number': '27',
    'company_region': '',
    'company_municipality': 'ΚΑΒΑΛΑΣ / ΚΑΒΑΛΑΣ',
    'company_zip_code': '65302',
    'companyWebsite': None,
    'companyEshop': None,
    'isABranch': False,
    'isForeignBranch': False,
    'lock': 0},
   'capital': [{'descr': None,
     'amount': None,
     'nominal_price': None,
     'capital_stock': 103000,
